In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [1]:
!pip install -q opencv-python-headless==4.9.0.80
!pip install -q ultralytics==8.0.149 torchcam albumentations==1.3.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.6/49.6 MB 28.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 614.4/614.4 kB 13.3 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 123.5/123.5 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.0/63.0 MB 24.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 84.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 61.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 36.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 27.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
import ultralytics, torchcam, albumentations, cv2
print("✅ All main packages imported successfully")

✅ All main packages imported successfully


In [3]:
import os
base = "/kaggle/input/vinbigdata-chest-xray-abnormalities-detection"
for dirname, _, filenames in os.walk(base):
    print(dirname, len(filenames))
    for f in filenames[:10]:
        print("  ", f)
    break

/kaggle/input/vinbigdata-chest-xray-abnormalities-detection 2
   sample_submission.csv
   train.csv


In [4]:
import pandas as pd, numpy as np
ann_path = os.path.join(base, "train.csv")
df = pd.read_csv(ann_path)
print("Rows (boxes):", len(df))
print("Unique images:", df["image_id"].nunique())
classes = sorted(df["class_name"].unique())
print("Classes:", classes)
display(df.head())
display(df.groupby("class_name").size().sort_values(ascending=False).rename("count").to_frame())

Rows (boxes): 67914
Unique images: 15000
Classes: ['Aortic enlargement', 'Atelectasis', 'Calcification', 'Cardiomegaly', 'Consolidation', 'ILD', 'Infiltration', 'Lung Opacity', 'No finding', 'Nodule/Mass', 'Other lesion', 'Pleural effusion', 'Pleural thickening', 'Pneumothorax', 'Pulmonary fibrosis']


/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in greater
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()


,image_id,class_name,class_id,rad_id,x_min,y_min,x_max,y_max
0,50a418190bc3fb1ef1633bf9678929b3,No finding,14,R11,NaN,NaN,NaN,NaN
1,21a10246a5ec7af151081d0cd6d65dc9,No finding,14,R7,NaN,NaN,NaN,NaN
2,9a5094b2563a1ef3ff50dc5c7ff71345,Cardiomegaly,3,R10,691.0,1375.0,1653.0,1831.0
3,051132a778e61a86eb147c7c6f564dfe,Aortic enlargement,0,R10,1264.0,743.0,1611.0,1019.0
4,063319de25ce7edb9b1c6b8881290140,No finding,14,R10,NaN,NaN,NaN,NaN


,count
class_name,
No finding,31818
Aortic enlargement,7162
Cardiomegaly,5427
Pleural thickening,4842
Pulmonary fibrosis,4655
Nodule/Mass,2580
Lung Opacity,2483
Pleural effusion,2476
Other lesion,2203


In [5]:
import shutil, os
IMG_SRC = os.path.join(base, "train")   # images folder in dataset
IMG_DST = "/kaggle/working/images_all"
os.makedirs(IMG_DST, exist_ok=True)
SUBSET = False        # set True for quick iteration
MAX_IMAGES = 2000

image_ids = sorted(df["image_id"].unique())
if SUBSET:
    image_ids = image_ids[:MAX_IMAGES]

copied = 0
for img_id in image_ids:
    src = os.path.join(IMG_SRC, f"{img_id}.jpg")
    dst = os.path.join(IMG_DST, f"{img_id}.jpg")
    if os.path.exists(src):
        shutil.copy(src, dst)
        copied += 1
print("Copied images:", copied, "to", IMG_DST)

Copied images: 0 to /kaggle/working/images_all


In [6]:
from pathlib import Path
from PIL import Image
lab_out = "/kaggle/working/labels_all"
os.makedirs(lab_out, exist_ok=True)

class_names = sorted(df["class_name"].unique())
cls2id = {c:i for i,c in enumerate(class_names)}
print("class mapping sample:", list(cls2id.items())[:8])

groups = df.groupby("image_id")
written = 0
for img_id, g in groups:
    img_path = os.path.join(IMG_DST, f"{img_id}.jpg")
    if not os.path.exists(img_path):
        continue
    w, h = Image.open(img_path).size
    lines = []
    for _, row in g.iterrows():
        x_min = row["x_min"]; y_min = row["y_min"]
        bw = row["width"]; bh = row["height"]
        x_c = (x_min + bw/2.0) / w
        y_c = (y_min + bh/2.0) / h
        w_n = bw / w; h_n = bh / h
        cls_id = cls2id[row["class_name"]]
        lines.append(f"{cls_id} {x_c:.6f} {y_c:.6f} {w_n:.6f} {h_n:.6f}")
    if lines:
        with open(os.path.join(lab_out, f"{img_id}.txt"), "w") as f:
            f.write("\n".join(lines))
        written += 1
print("Label files written:", written, "to", lab_out)

# Save class names for reference
with open("/kaggle/working/classes.txt", "w") as f:
    f.write("\n".join(class_names))

class mapping sample: [('Aortic enlargement', 0), ('Atelectasis', 1), ('Calcification', 2), ('Cardiomegaly', 3), ('Consolidation', 4), ('ILD', 5), ('Infiltration', 6), ('Lung Opacity', 7)]
Label files written: 0 to /kaggle/working/labels_all


In [7]:
import random
random.seed(42)
from pathlib import Path
img_list = sorted([p.stem for p in Path(IMG_DST).glob("*.jpg")])
random.shuffle(img_list)
n = len(img_list)
ntrain = int(0.7 * n)
nval = int(0.15 * n)
train_ids = img_list[:ntrain]
val_ids = img_list[ntrain:ntrain+nval]
test_ids = img_list[ntrain+nval:]

for split, ids in [("train", train_ids), ("val", val_ids), ("test", test_ids)]:
    imdir = f"/kaggle/working/images/{split}"
    lbdir = f"/kaggle/working/labels/{split}"
    os.makedirs(imdir, exist_ok=True)
    os.makedirs(lbdir, exist_ok=True)
    for img_id in ids:
        src_img = os.path.join(IMG_DST, f"{img_id}.jpg")
        dst_img = os.path.join(imdir, f"{img_id}.jpg")
        if os.path.exists(src_img):
            shutil.copy(src_img, dst_img)
        src_lb = os.path.join(lab_out, f"{img_id}.txt")
        dst_lb = os.path.join(lbdir, f"{img_id}.txt")
        if os.path.exists(src_lb):
            shutil.copy(src_lb, dst_lb)

print("train/val/test counts:", len(train_ids), len(val_ids), len(test_ids))

# write dataset.yaml for ultralytics
import yaml
dataset_yaml = {
    "path": "/kaggle/working",
    "train": "images/train",
    "val": "images/val",
    "test": "images/test",
    "names": {i: n for i,n in enumerate(class_names)}
}
with open("/kaggle/working/dataset.yaml", "w") as f:
    yaml.dump(dataset_yaml, f)
print("Wrote /kaggle/working/dataset.yaml")

train/val/test counts: 0 0 0
Wrote /kaggle/working/dataset.yaml


In [8]:
import os, glob, pprint

print("IMG_SRC (original input):", "/kaggle/input/vinbigdata-chest-xray-abnormalities-detection/train")
print("IMG_DST (copied):", "/kaggle/working/images_all")
print("Labels all:", "/kaggle/working/labels_all")
print("Train images folder:", "/kaggle/working/images/train")
print("Train labels folder:", "/kaggle/working/labels/train")
print()

def show_counts(path, pattern="*.jpg"):
    if not os.path.exists(path):
        print(f"  {path} -> DOES NOT EXIST")
        return 0
    cnt = len(glob.glob(os.path.join(path, pattern)))
    print(f"  {path} -> {cnt}")
    return cnt

show_counts("/kaggle/input/vinbigdata-chest-xray-abnormalities-detection/train", "*.jpg")
show_counts("/kaggle/working/images_all", "*.jpg")
show_counts("/kaggle/working/labels_all", "*.txt")
show_counts("/kaggle/working/images/train", "*.jpg")
show_counts("/kaggle/working/labels/train", "*.txt")
show_counts("/kaggle/working/images/val", "*.jpg")
show_counts("/kaggle/working/labels/val", "*.txt")
show_counts("/kaggle/working/images/test", "*.jpg")
show_counts("/kaggle/working/labels/test", "*.txt")

# Print a few file names from each place to inspect
print("\nSample files (up to 10):")
for p in ["/kaggle/working/images_all", "/kaggle/working/labels_all", "/kaggle/working/images/train"]:
    if os.path.exists(p):
        print(f"\n{p}:")
        for f in sorted(os.listdir(p))[:10]:
            print("  ", f)
    else:
        print(f"\n{p}: DOES NOT EXIST")

IMG_SRC (original input): /kaggle/input/vinbigdata-chest-xray-abnormalities-detection/train
IMG_DST (copied): /kaggle/working/images_all
Labels all: /kaggle/working/labels_all
Train images folder: /kaggle/working/images/train
Train labels folder: /kaggle/working/labels/train

  /kaggle/input/vinbigdata-chest-xray-abnormalities-detection/train -> 0
  /kaggle/working/images_all -> 0
  /kaggle/working/labels_all -> 0
  /kaggle/working/images/train -> 0
  /kaggle/working/labels/train -> 0
  /kaggle/working/images/val -> 0
  /kaggle/working/labels/val -> 0
  /kaggle/working/images/test -> 0
  /kaggle/working/labels/test -> 0

Sample files (up to 10):

/kaggle/working/images_all:

/kaggle/working/labels_all:

/kaggle/working/images/train:


In [10]:
import os, pathlib, pydicom, numpy as np, cv2, tqdm, shutil

SRC = "/kaggle/input/vinbigdata-chest-xray-abnormalities-detection/train"
DST = "/kaggle/working/images_all"
os.makedirs(DST, exist_ok=True)

SUBSET = True   # set False to convert ALL images (15000). True for debug.
MAX_IMAGES = 3000  # when subset True, convert up to this many

all_dcmins = sorted([str(p) for p in pathlib.Path(SRC).glob("*.dicom")] + [str(p) for p in pathlib.Path(SRC).glob("*.dcm")])
print("Found DICOMs:", len(all_dcmins))
if SUBSET:
    all_dcmins = all_dcmins[:MAX_IMAGES]
    print("Using subset, converting:", len(all_dcmins))

def read_dicom_to_np(path):
    try:
        ds = pydicom.dcmread(path)
        arr = ds.pixel_array.astype(np.float32)
        # PhotometricInterpretation handling
        if getattr(ds, "PhotometricInterpretation", "").upper() == "MONOCHROME1":
            arr = arr.max() - arr
        # Some DICOMs have rescale intercept & slope
        intercept = getattr(ds, "RescaleIntercept", 0.0)
        slope = getattr(ds, "RescaleSlope", 1.0)
        arr = arr * slope + intercept
        return arr
    except Exception as e:
        print("Failed read:", path, e)
        return None

def normalize_to_uint8(img):
    # Clip extreme percentiles to reduce outliers then scale
    p1, p99 = np.percentile(img, (1, 99))
    img = np.clip(img, p1, p99)
    img = img - img.min()
    if img.max() > 0:
        img = img / img.max()
    img_u8 = (img * 255.0).astype(np.uint8)
    return img_u8

count = 0
for dpath in tqdm.tqdm(all_dcmins):
    arr = read_dicom_to_np(dpath)
    if arr is None:
        continue
    img = normalize_to_uint8(arr)
    # resize long edge to 1024 for storage (optional)
    h,w = img.shape[:2]
    max_edge = 1024
    if max(h,w) > max_edge:
        scale = max_edge / max(h,w)
        img = cv2.resize(img, (int(w*scale), int(h*scale)), interpolation=cv2.INTER_AREA)
    # save as jpg
    out_name = os.path.splitext(os.path.basename(dpath))[0] + ".jpg"
    out_path = os.path.join(DST, out_name)
    cv2.imwrite(out_path, img)
    count += 1

print("Converted images saved to:", DST, "total:", count)

Found DICOMs: 15000
Using subset, converting: 3000


100%|██████████| 3000/3000 [1:00:45<00:00,  1.22s/it]

Converted images saved to: /kaggle/working/images_all total: 3000


In [11]:
import os, pathlib, traceback
import pydicom, numpy as np, cv2

SRC = "/kaggle/input/vinbigdata-chest-xray-abnormalities-detection/train"
files = sorted([str(p) for p in pathlib.Path(SRC).glob("*")])[:20]
print("First files (up to 20) in train folder (showing names and sizes):")
for f in files[:20]:
    try:
        print(" ", os.path.basename(f), "-", os.path.getsize(f), "bytes")
    except Exception as e:
        print(" ", os.path.basename(f), "- size error:", e)

print("\nAttempt to read up to 5 DICOM files and inspect pixel data:")
count = 0
for f in files:
    if count >= 5:
        break
    try:
        ds = pydicom.dcmread(f, force=True)
        print(f"\nFile: {os.path.basename(f)}")
        # print some header info if present
        for tag in ["PatientID", "StudyInstanceUID", "SeriesInstanceUID", "Rows", "Columns", "PhotometricInterpretation", "BitsAllocated", "BitsStored", "PixelRepresentation"]:
            try:
                val = getattr(ds, tag)
            except Exception:
                val = None
            print("  ", tag, "=", val)
        # try to access pixel_array
        try:
            arr = ds.pixel_array
            print("  pixel_array shape:", getattr(arr, "shape", None), "dtype:", getattr(arr, "dtype", None))
            # basic normalization test
            arrf = arr.astype("float32")
            p1, p99 = np.percentile(arrf, (1,99))
            arrf = np.clip(arrf, p1, p99)
            arrf = arrf - arrf.min()
            if arrf.max() > 0:
                arrf = (arrf / arrf.max() * 255.0).astype("uint8")
            else:
                arrf = arrf.astype("uint8")
            out_path = f"/kaggle/working/tmp_test_{count}.jpg"
            cv2.imwrite(out_path, arrf)
            print("  wrote test jpg:", out_path, "size:", os.path.getsize(out_path))
        except Exception as e:
            print("  could not access pixel_array or write jpg. Exception:")
            traceback.print_exc()
    except Exception as e:
        print("\nFailed to read DICOM:", f)
        traceback.print_exc()
    count += 1

print("\nDiagnostic complete. Check /kaggle/working for tmp_test_*.jpg files.")
print("Listing /kaggle/working jpgs:")
import glob
print(glob.glob("/kaggle/working/tmp_test_*.jpg"))

First files (up to 20) in train folder (showing names and sizes):
  000434271f63a053c4128a0ba6352c7f.dicom - 13250272 bytes
  00053190460d56c53cc3e57321387478.dicom - 9691322 bytes
  0005e8e3701dfb1dd93d53e2ff537b6e.dicom - 18874874 bytes
  0006e0a85696f6bb578e84fafa9a5607.dicom - 8021776 bytes
  0007d316f756b3fa0baea2ff514ce945.dicom - 13271518 bytes
  000ae00eb3942d27e0b97903dd563a6e.dicom - 18874882 bytes
  000d68e42b71d3eac10ccc077aba07c1.dicom - 6609510 bytes
  00150343289f317a0ad5629d5b7d9ef9.dicom - 12665874 bytes
  00176f7e1b1cb835123f95960b9a9efd.dicom - 16750760 bytes
  001d127bad87592efe45a5c7678f8b8d.dicom - 18874874 bytes
  0021df30f3fddef551eb3df4354b1d06.dicom - 18874874 bytes
  00291f7aff0123ea76a59998effef229.dicom - 18874874 bytes
  0032c6091dc8f1b1245fc2f5f45458fa.dicom - 6639402 bytes
  003cfe5ce5c0ec5163138eb3b740e328.dicom - 9691322 bytes
  0046f681f078851293c4e710c4466058.dicom - 6790470 bytes
  004d2bc2111d639f5e8441ced52d55cb.dicom - 4972598 bytes
  004dc2a5059